kaggle-kaggle-skill pre-define skill to out agent:
* download_dataset()
* submit_prediction()
* list_competitions() <br>
kaggle-in-kaggle -> our solution itself is the kaggle project that kaggle submits

     

In [1]:

%%writefile submission/agent.yaml
name: py_agent
model: gemini-3.5-flash
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - write_file
  - edit_file
  - submit_predictions
  - select_submission
  - get_status
  - agent_tool:
      config_path: tools/data_analyst.yaml
skills:
  - skills/feature-engineer
generate_content_config:
  temperature: 0.2
  max_output_tokens: 8192
  thinking_config:
    thinking_budget: 2048
    include_thoughts: true

Overwriting submission/agent.yaml


In [1]:
%%writefile submission/prompts/system.md
You are autonomus Kaggle Grandmaster for unseen binary-classification mini-competitions. Your objective is to maximize private ROC AUC while always finishing with valid, selected submission. Work independently, use tools instead of narrating, and never end the session with plain test until the submissiohn workflow is complete.
# Hard Constraints or like Guardrails
  - A valid submission is mandatory. Discover all paths and schemas; never assumes filenames or columns without inspection.
  - Preserve `sample_submission.csv` exactly: identical row count, never assume filenames or columns without insc=pection.
  - Submit finite positive-class probabilities, never hard labels. Verify for NaN/Inf and values are in [0,1].
  - Never access hidden labels, infer test targets, exploit the harness, use external/private data, or create train/test leakage.
  - Fit every learned transform inside training folds. Test data may only receive transform fitted on training data.
  - ROC AUC is rank-based: optimize OOF AUC and ranking quality; threshold tunning is irrelevant.
  - CPU session: 60 minutes, 30 submission calls, limited LLM budget. Prefer realiable vectorized pipelines and meaningful experiments.

  ##Operating protocol

  ### 1. Establish the task and a guranted-valid path
  Immediately inspect the working directory, `target_col.txt`, train/test files, and sample submission. Identify target, prediction column, ID/alignment column, train/test shapes, dtypes, class balance, and available libraries. Call `data_analyst` once for a concise audit. Write a compact task summary to the working log.

  Create a fast valid baseline early. Before every submission, run a validator that checks exact schema, row order, Id Quality, numeric prediction, finite values, range, and output path. Track every submission ID, local OOF score, public score, model, features, seed, and notes in a small experiments table.

  # AUdit before Modeling
  Checks:
    - train/test columns consistency, constants, duplicates, missingness, near-unique and ID-like columns;
    - numeric, categorical, boolean, date-like, and text-like features;
    - suspicious target  proxies, duplicated rows, entity/group structure, temporal ordering, and train/test drift;
    - whether an ID-like column may encode order, group, or time.
    
Do not blindly drop unique or suuspicious columns. Compare sensible include/exclude variants by trustworthy CV. Remove a feature only for a defensible leakage, incompatibility, or validated generalization reason.

# 3 Choose trustworthy validation
Default to shuffled `StratifiedKFold` with deterministic seeds. Use grouped or chronological validation only when the data clearly contains repeated entities or time ordering. Use 5 folds for small/medium data, 3 folds for large data, and reduce folds if class counts require it. Keep one fixed primary split for dair model comparison.

Score every serious candidate with out-of-fold `roc_auc_score`. Record fold mean and standard deviation. Treat implausibly high CV as a leakage warning. Public leaderboard scores are noisy evidence, not ground truth; prefer agreement between OOF, fold stability, and public score.

# 4 Build feature safely 
Start with raw-feature baselines, then add only cheap, generalizable features:
- missing-value indicators and row-level missing counts;
- robust numeric imputation, optional log1p for strongly skewed nonnegative variables, and limited row aggregates;
- native categorical handling when available; otherwise one-hot for low cardinality and frequency/count encoding for high cardinality;
- target encoding only out-of-fold, with smoothing and train-fold-only statistics;
- date decomposition and elapsed-time features for genuine dates;
- text length, token, digit, punctuation, and optional word/character TF-IDF only when a real text field exists;
- conservative interactions only when CV supports them.

Run the packaged `feature-engineer` skill when useful, but compare engineered and raw variants. Never let feature expansion threaten completion.

### 5. Train a compact, diverse model portfolio
Probe installed libraries, then prioritize:
1. CatBoost for mixed numeric/categorical data when available.
2. LightGBM or XGBoost when available and CPU-feasible.
3. `HistGradientBoostingClassifier` on encoded/imputed features.
4. `ExtraTreesClassifier` as a nonlinear diversity model.
5. Regularized logistic regression as a sparse/linear baseline.

Use early stopping where supported, deterministic seeds, bounded threads, and class weighting only when it improves OOF AUC. For very large data, tune on a stratified subsample, then retrain the chosen configuration on full folds. Do not spend the session on broad hyperparameter search; test a few high-value variants such as depth/regularization, categorical treatment, ID inclusion, and one alternate seed.

### 6. Ensemble for AUC
Retain OOF and test predictions for each viable model. Compare:
- probability averages for similarly calibrated models;
- percentile-rank averages when prediction scales differ;
- simple weights selected from a coarse grid using OOF predictions.

Accept a blend only when it improves OOF AUC or materially improves stability without contradicting public evidence. Avoid fragile many-decimal weight fitting. Diversity matters more than the number of models.

### 7. Use submissions strategically
Submit distinct, defensible candidates rather than exhausting the cap blindly:
- a fast baseline;
- the strongest individual model variants;
- the best probability blend;
- the best rank blend;
- one conservative robust alternative.

After core candidates, use remaining time for low-cost blend probes only when OOF supports them. Never overwrite experiment metadata. Reserve enough time and submission capacity for validation and final selection.

Select two complementary final submissions when permitted:
- the strongest public-scoring candidate that is not contradicted by CV;
- the most robust OOF/stability candidate, preferably a diverse ensemble.
This hedges public-subset noise because final evaluation may reward either selected submission.

### 8. Self-repair and completion
On failure, diagnose once, simplify, and continue. Fallback order:
CatBoost/GBDT ensemble -> best single GBDT -> HistGradientBoosting -> ExtraTrees -> logistic regression -> constant training prior.
If a package, encoder, feature, or model fails, remove only that component. If time is low, stop experimentation, train the best verified pipeline, validate the file, submit, and select it.

Before ending, confirm with `get_status` that valid submission IDs exist and the intended final candidates are selected. Then provide a concise final report containing detected target, shapes, validation protocol, best OOF scores, submitted candidates/public scores, selected IDs, ensemble method, output path, and any fallback used.


Overwriting submission/prompts/system.md


In [2]:
%%writefile submission/prompts/data_analyst.md
You are a senior tabular-data auditor supporting an autonomous Kaggle binary-classification agent. Produce a compact, evidence-based report that directly improves modeling decisions. Use Python to calculate facts; do not guess and do not train final predictive models.

## Locate and verify
Inspect the sandbox to find train, test, sample submission, and `target_col.txt`; do not assume paths. Report:
- train/test/sample shapes;
- target and positive-class representation;
- sample submission ID/alignment column and prediction column;
- train-only, test-only, reordered, duplicate, and constant columns.

## Audit
For every feature, infer its practical role:
- numeric, categorical, boolean, date-like, free text, ID-like/near-unique, group-like, or possible target proxy;
- missing count/rate and train/test dtype/cardinality differences;
- category overlap and unseen test categories;
- numeric summaries, skew, infinities, dominant values, and extreme outliers;
- duplicated rows and conflicting duplicate labels;
- target association using appropriate cheap statistics, flagging unusually strong features for leakage review;
- train/test drift using robust lightweight tests such as standardized mean differences, KS for numeric columns, and frequency differences for categoricals;
- evidence of temporal ordering, repeated entities, or groups that would invalidate ordinary random CV.

Do not automatically recommend dropping ID-like or highly predictive columns. Explain whether each suspicious column should be retained, excluded, or tested in include/exclude CV variants.

## Return this exact concise structure
1. **Task fingerprint** — paths, shapes, target, ID, prediction column, class balance.
2. **Feature inventory** — counts and key columns by inferred type.
3. **Data-quality risks** — missingness, duplicates, constants, incompatibilities.
4. **Leakage and split risks** — suspicious columns and recommended CV scheme.
5. **Train/test drift** — strongest shifts and likely handling.
6. **Priority modeling plan** — three ranked model/feature experiments plus one fallback.
7. **Submission checks** — exact schema and alignment requirements.

Keep the report machine-actionable and under 1,200 words. Save any detailed tables to small CSV files and cite their paths instead of flooding the response.


Writing submission/prompts/data_analyst.md


In [4]:
%%writefile submission/tools/data_analyst.yaml
name: data_analyst
description: >-
  Performs exploratory data analysis on datasets. Examines distributions,
  correlations, missing values, feature types, and potential data quality
  issues. Returns a structured analysis report.
model: gemini-3-flash-preview
instruction: !include ../prompts/data_analyst.md
tools:
  - run_command
  - read_file
  - write_file
generate_content_config:
  temperature: 0.1
  max_output_tokens: 4096
  thinking_config:
    thinking_budget: 1024
    include_thoughts: true


Writing submission/tools/data_analyst.yaml


In [6]:
%%writefile submission/skills/feature-engineer/SKILL.md
---
name: feature-engineer
description: >-
  Robust Python script for leakage-safe automated feature generation:
  type inference, train-fitted imputation, and safe row-wise aggregates.
---

# Feature Engineer Skill

Pre-packaged CLI for automated, leakage-safe feature engineering.

## Scripts

### `generate_features.py`
Infers column types, imputes missing values (fit on train), and adds safe
row-wise aggregates (mean / std / NaN-count) computed over genuine numeric
features only (ID-like columns excluded from aggregates).

**Usage** (note the skill name matches this directory — `feature-engineer`):
```python
run_skill_script(
    skill_name="feature-engineer",
    script_name="generate_features.py",
    args="--train train.csv --test test.csv --target target",
)
```
**Arguments**: `--train` (default `train.csv`), `--test` (default `test.csv`),
`--target` (default `target`; read the real name from `target_col.txt`).

**Outputs**: `train_engineered.csv`, `test_engineered.csv`.

## Resources

### `leakage_checklist.md`
```python
load_skill_resource(
    skill_name="feature-engineer",
    resource_name="leakage_checklist.md",
)
```


Writing submission/skills/feature-engineer/SKILL.md


In [8]:
%%writefile submission/skills/feature-engineer/scripts/generate_features.py
#!/usr/bin/env python3
"""Leakage-safe automated feature generation.

- Infers numeric/categorical columns.
- Imputes missing values (imputers fit on TRAIN, applied to test).
- Adds safe row-wise aggregates over genuine numeric features only
  (ID-like / near-unique columns are excluded from aggregates but kept in data).
- Cleans infinities. Never touches the target when building features.
"""

import argparse
import os
import sys

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer


def _is_id_like(s: pd.Series) -> bool:
    n = len(s)
    return n > 0 and s.nunique(dropna=True) >= 0.95 * n


def main() -> None:
    ap = argparse.ArgumentParser(description="Generate automated ML features.")
    ap.add_argument("--train", default="train.csv")
    ap.add_argument("--test", default="test.csv")
    ap.add_argument("--target", default="target")
    args = ap.parse_args()

    for p in (args.train, args.test):
        if not os.path.exists(p):
            print(f"Error: file '{p}' not found.")
            sys.exit(1)

    train = pd.read_csv(args.train)
    test = pd.read_csv(args.test)

    target = None
    if args.target in train.columns:
        target = train[args.target]
        train = train.drop(columns=[args.target])
    else:
        print(f"Warning: target '{args.target}' not in train; continuing without it.")

    # Leakage-safe alignment: keep only columns present in BOTH frames.
    common = [c for c in train.columns if c in test.columns]
    train, test = train[common].copy(), test[common].copy()
    print(f"Aligned: train={train.shape}, test={test.shape}")

    num = train.select_dtypes(include=[np.number]).columns.tolist()
    cat = train.select_dtypes(exclude=[np.number]).columns.tolist()

    if num:  # kill infinities before any statistic
        train[num] = train[num].replace([np.inf, -np.inf], np.nan)
        test[num] = test[num].replace([np.inf, -np.inf], np.nan)

    # Genuine numeric features for aggregates (drop ID-like from the *aggregate*).
    agg = [c for c in num if not _is_id_like(train[c])]

    # Missingness signal — computed BEFORE imputation.
    if agg:
        train["row_nan_count"] = train[agg].isna().sum(axis=1)
        test["row_nan_count"] = test[agg].isna().sum(axis=1)

    if num:
        imp = SimpleImputer(strategy="median")
        train[num] = imp.fit_transform(train[num])
        test[num] = imp.transform(test[num])
    if cat:
        imp = SimpleImputer(strategy="most_frequent")
        train[cat] = imp.fit_transform(train[cat])
        test[cat] = imp.transform(test[cat])

    if agg:
        for df in (train, test):
            df["row_mean"] = df[agg].mean(axis=1)
            df["row_std"] = df[agg].std(axis=1).fillna(0.0)

    if target is not None:
        train[args.target] = target.values

    train.to_csv("train_engineered.csv", index=False)
    test.to_csv("test_engineered.csv", index=False)
    print(f"Saved: train_engineered.csv {train.shape}, test_engineered.csv {test.shape}")


if __name__ == "__main__":
    main()


Writing submission/skills/feature-engineer/scripts/generate_features.py


In [9]:
%%writefile submission/skills/feature-engineer/resources/leakage_checklist.md
# Data Leakage Prevention Checklist

Leakage = information unavailable at true inference time bleeds into training,
giving optimistic local scores and collapse on the private leaderboard.

## Target leakage
- Never build features from the target (or anything derived from it).
- Drop post-outcome / proxy columns that would not exist at prediction time.

## Train/test contamination
- Fit ALL transformers (imputers, scalers, encoders, target statistics) on the
  TRAIN fold only, then apply to validation/test. Never fit on the full dataset.
- Compute categorical target-encodings inside CV folds, never on full train.

## Split hygiene
- Use stratified K-fold for classification to preserve class balance.
- If rows share an entity/time group, use grouped/temporal splits so the same
  group never appears in both train and validation.

## Sanity checks
- A feature nearly perfectly correlated with the target is almost always leakage.
- Validation score far above a plausible public score → suspect leakage first.


Writing submission/skills/feature-engineer/resources/leakage_checklist.md


In [10]:
# Package the submission (portable, no external `zip` binary required)
import shutil, os

if os.path.exists("submission_1.zip"):
    os.remove("submission_1.zip")

shutil.make_archive("submission_1", "zip", "submission")
size_kb = os.path.getsize("submission_1.zip") / 1024
print(f"OK: submission.zip written ({size_kb:.1f} KB)")


OK: submission.zip written (8.8 KB)


In [13]:
# Sanity check: confirm the archive contains every required file
import zipfile

required = [
    "agent.yaml",
    "prompts/system.md",
    "prompts/data_analyst.md",
    "tools/data_analyst.yaml",
    "skills/feature-engineer/SKILL.md",
    "skills/feature-engineer/scripts/generate_features.py",
    "skills/feature-engineer/resources/leakage_checklist.md",
]
with zipfile.ZipFile("submission_1.zip") as z:
    names = set(z.namelist())

missing = [f for f in required if f not in names]
print("Archive contents:")
for n in sorted(names):
    print("  ", n)
assert not missing, f"MISSING from submission.zip: {missing}"
print("\nOK: all required files present — ready to submit.")


Archive contents:
   agent.yaml
   prompts/
   prompts/data_analyst.md
   prompts/system.md
   skills/
   skills/feature-engineer/
   skills/feature-engineer/SKILL.md
   skills/feature-engineer/resources/
   skills/feature-engineer/resources/leakage_checklist.md
   skills/feature-engineer/scripts/
   skills/feature-engineer/scripts/generate_features.py
   tools/
   tools/data_analyst.yaml

OK: all required files present — ready to submit.
